In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg") 
import matplotlib.pyplot as plt
import os 

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score, precision_recall_curve,
    log_loss, classification_report
)


plots_dir = "plots/"
os.makedirs(plots_dir, exist_ok=True)


print("1) BINARY CLASSIFICATION — Breast Cancer dataset (malignant vs benign)")


data = load_breast_cancer()
X, y = data.data, data.target  # 0 = malignant, 1 = benign
print("Classes:", data.target_names, "| Shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
y_scores = clf.predict_proba(X_test_s)[:, 1]  # probability of class "1" (benign)

# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot()
plt.title("Confusion Matrix — Breast Cancer")
plt.savefig(os.path.join(plots_dir, "20_confusion_matrix.png"), bbox_inches="tight")
plt.close()

# ---- Core metrics ----
print("\nAccuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1-score :", round(f1_score(y_test, y_pred), 4))
print("\nFull classification report:\n", classification_report(y_test, y_pred, target_names=data.target_names))

# ---- ROC-AUC ----
roc_auc = roc_auc_score(y_test, y_scores)
fpr, tpr, roc_thresholds = roc_curve(y_test, y_scores)
print("ROC-AUC  :", round(roc_auc, 4))

plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(os.path.join(plots_dir, "20_roc_curve.png"), bbox_inches="tight")
plt.close()

# ---- PR-AUC ----
pr_auc = average_precision_score(y_test, y_scores)
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_scores)
print("PR-AUC (Average Precision):", round(pr_auc, 4))

plt.figure()
plt.plot(recall_vals, precision_vals, label=f"PR curve (AP = {pr_auc:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", color="gray", label="No-skill baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.savefig(os.path.join(plots_dir, "20_pr_curve.png"), bbox_inches="tight")
plt.close()

# ---- Log Loss ----
ll = log_loss(y_test, y_scores)
print("Log Loss :", round(ll, 4))

# ---- Threshold tuning ----
print("\n--- Threshold tuning: maximize F1 ---")
f1_scores = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-9)
best_idx = np.argmax(f1_scores[:-1])  # last point has no matching threshold
best_threshold = pr_thresholds[best_idx]
print(f"Best threshold for F1: {best_threshold:.3f}  (F1 = {f1_scores[best_idx]:.4f})")

y_pred_default = (y_scores >= 0.5).astype(int)
y_pred_tuned = (y_scores >= best_threshold).astype(int)
print(f"F1 @ default threshold 0.5   : {f1_score(y_test, y_pred_default):.4f}")
print(f"F1 @ tuned threshold {best_threshold:.3f}   : {f1_score(y_test, y_pred_tuned):.4f}")


print("2) MULTICLASS CLASSIFICATION — Wine dataset (3 classes)")


wine = load_wine()
Xw, yw = wine.data, wine.target
print("Classes:", wine.target_names, "| Shape:", Xw.shape)

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.2, stratify=yw, random_state=42
)
scaler_w = StandardScaler()
Xw_train_s = scaler_w.fit_transform(Xw_train)
Xw_test_s = scaler_w.transform(Xw_test)

clf_multi = LogisticRegression(max_iter=5000)  # sklearn now auto-selects multinomial for multiclass
clf_multi.fit(Xw_train_s, yw_train)
yw_pred = clf_multi.predict(Xw_test_s)

cm_multi = confusion_matrix(yw_test, yw_pred)
print("\nMulticlass Confusion Matrix:\n", cm_multi)
print("\nAccuracy (multiclass):", round(accuracy_score(yw_test, yw_pred), 4))
print("\nPer-class report:\n", classification_report(yw_test, yw_pred, target_names=wine.target_names))

# Macro vs weighted vs micro averaging
print("F1 macro   :", round(f1_score(yw_test, yw_pred, average="macro"), 4))
print("F1 weighted:", round(f1_score(yw_test, yw_pred, average="weighted"), 4))
print("F1 micro   :", round(f1_score(yw_test, yw_pred, average="micro"), 4))

print("\nSaved plots ")
print("Done.")

1) BINARY CLASSIFICATION — Breast Cancer dataset (malignant vs benign)
Classes: ['malignant' 'benign'] | Shape: (569, 30)

Confusion Matrix:
 [[41  1]
 [ 1 71]]
TN=41  FP=1  FN=1  TP=71

Accuracy : 0.9825
Precision: 0.9861
Recall   : 0.9861
F1-score : 0.9861

Full classification report:
               precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

ROC-AUC  : 0.9954
PR-AUC (Average Precision): 0.9971
Log Loss : 0.0777

--- Threshold tuning: maximize F1 ---
Best threshold for F1: 0.366  (F1 = 0.9931)
F1 @ default threshold 0.5   : 0.9861
F1 @ tuned threshold 0.366   : 0.9931
2) MULTICLASS CLASSIFICATION — Wine dataset (3 classes)
Classes: ['class_0' 'class_1' 'class_2'] | Shape: (178, 13)

Multiclass Confusion Matrix:
 [[12  0